# ParkWise Nairobi: Data Pipeline v2

This notebook builds the facility, pricing, traffic, and calibration layers used by the
ParkWise Nairobi Availability Predictor. Every cell is written to make one thing clear
at all times: which numbers are real (pulled from OpenStreetMap, the TomTom Traffic API,
or a genuine field observation), which are documented estimates (like the NCCG placeholder
rate), and which are still missing. No column in the output of this notebook should be
read as measured parking occupancy unless it comes from `nairobi_parking_spotcheck.csv`
or from `parking_pressure_score` after real calibration.


## Business Overview

Nairobi has a well documented parking shortage. A 2011 IBM commuter survey found that
motorists in Nairobi spend an average of 31.7 minutes searching for a parking spot,
compared with a 19.8 minute global average measured in the same study. A 2019 National
Assembly Public Accounts Committee report found that Nairobi, a city with more than 1.3
million registered vehicles, had only around 6,125 formal parking spaces, most of them
operated by Nairobi City County.

The county has since digitised parking fee payment through platforms such as
nairobiservices.go.ke and the Nairobi Pay app, and a growing number of private ventures,
including Spot Finder, Parksby, Naiparq, and KERB, now offer app based parking discovery
and payment across zones such as the CBD, Westlands, Kilimani, and Upper Hill. What none
of these platforms currently do, based on their public descriptions, is tell a driver
what will be available at a future time. They report what is listed, bookable, or
reportedly open right now, not what is likely to be available later.

ParkWise Nairobi is a data driven prediction platform that lets a driver check, ahead of
time, whether parking is likely to be available at a chosen location for a future date
and time, whether that is later the same day or a week ahead. The prediction is built
from patterns in historical demand signals, time of day, day of week, and holiday
effects, rather than only reporting current state.


## Problem Statement

Drivers in high demand Nairobi areas cannot currently plan a trip around parking
availability in advance. They discover whether space exists only by arriving and
circling, or by checking an app that reports current occupancy rather than a forecast.
This wastes time and fuel and adds to congestion in already constrained areas, and it
leaves both drivers and facility operators reacting to parking pressure rather than
anticipating it.

The central data challenge, and the reason this notebook is documented as closely as it
is, is that no public, official occupancy dataset exists for Nairobi parking facilities.
Cities with an open, sensor based occupancy feed can train a prediction model directly
against measured ground truth. Nairobi has no equivalent. The project team must either
collect that ground truth directly through field observation, or build the model around
the closest defensible proxy for real demand, while being explicit at every step about
which numbers are measured, which are calibrated estimates, and which are still missing.

Every data field produced by the pipeline below is labelled with its source, so that a
reader of the final report, or a teammate picking up this notebook later, can tell at a
glance whether a number reflects something that actually happened in Nairobi or something
the pipeline estimated in its absence.


## Data Understanding

### Overview of data sources

| Source | What it provides | Nature of the data |
|---|---|---|
| OpenStreetMap, via the Overpass API | Facility location, type, and whatever tags OSM contributors have added (name, fee, capacity, hours) | Real, but sparse. Most Nairobi parking facilities in OSM have no name and no capacity figure. |
| Nairobi City County Tariffs and Pricing Policy 2025-2030 and Finance Act 2023 | Zone I and Zone II on-street daily rates | Real, published rates, applied only to facilities that can be confirmed as on-street |
| Facility name text | Operator identity for off-street facilities (hospital, hotel, mall, church, ministry, and so on) | Real where a name is present, but the operator's own fee schedule is not published anywhere in this pipeline |
| TomTom Traffic Flow API | Live road congestion at a facility's coordinates | Real, live signal, but it measures road traffic, not parking occupancy |
| Field spot-check observations | Actual counted occupancy at a facility, at a specific date and time | Real ground truth, but limited to whatever the team has physically gone out and counted. All observations so far are at off-street facilities. |
| ITDP / University of Nairobi 2016 CBD Parking Survey | Published on-street bay counts and peak occupancy (~91-93%) for the surveyed CBD core | Real, professionally collected, but from 2016 and geographically scoped to a ~1.2 km CBD radius, not all 93 on-street facilities in this dataset |
| Google Maps reviews | Review text and star ratings for named facilities | Not yet available. The scraping attempt for this project has not produced usable data |

Each stage of the pipeline below is documented in its own markdown cell, immediately
before the code that implements it, covering what that cell does and why it matters for
the honesty of the final dataset. The closing cell of this notebook prints a data quality
summary that can be quoted directly in the project's data limitations section.

### Known limitations to keep in mind while reading this notebook

- OpenStreetMap's Nairobi parking coverage is sparse. Most facilities have no name and no
  reported capacity.
- Verified parking rates are only available for on-street facilities under the county's
  own tariff schedule. Off-street rates are placeholder estimates unless separately
  confirmed.
- Traffic congestion is a proxy for demand pressure, not a direct measurement of parking
  occupancy, and should never be reported as an occupancy figure on its own.
- Any conclusion drawn from `parking_pressure_score` is only as reliable as the spot-check
  sample it was calibrated against.
- Review based sentiment scoring has not yet produced any real data, and the dataset
  reflects this through missing values rather than placeholder scores.
- The TomTom free tier only provides live conditions from the point of each request
  onward, with no backfill for earlier dates, which is why running the snapshot logger
  early and often matters.
- The 93 on-street facilities are tiered by real distance to the CBD centre: `tier1_cbd`
  (21 facilities, within 1.2 km) carries the ITDP 2016 historical reference; `tier2_other`
  (72 facilities) does not, since that reference was never measured there.
- As of this writing, `parking_pressure_score` is empty for all 93 on-street facilities.
  Every real spot-check observation collected so far is at an off-street facility, so
  there is no real ground truth yet to calibrate the on-street tier against.


## Environment setup: loading the TomTom API key safely

This notebook reads `TOMTOM_API_KEY` from the environment, never from a value typed
directly into a cell. The cell below loads it from a local `.env` file using
`python-dotenv`, so the key lives in one gitignored file instead of in the notebook
itself.

Before running this cell:

1. Run `pip install python-dotenv` once in this environment.
2. Create a file named `.env` in this same folder, containing one line:
   `TOMTOM_API_KEY=your_actual_key_here`
3. Add `.env` to a `.gitignore` file in this folder so it is never committed.

Restart the kernel after creating `.env` for the first time, then run this cell before
any cell that calls the TomTom API.


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads the .env file in this folder into the environment, if present

TOMTOM_API_KEY = os.environ.get("TOMTOM_API_KEY", "")

if TOMTOM_API_KEY:
    print("TOMTOM_API_KEY loaded from environment. Live traffic calls are enabled.")
else:
    print(
        "TOMTOM_API_KEY not found.\n"
        "Check that a '.env' file exists in this folder with a line like "
        "TOMTOM_API_KEY=your_actual_key_here, then restart the kernel and re-run this cell."
    )


TOMTOM_API_KEY loaded from environment. Live traffic calls are enabled.


In [4]:
import pandas as pd
import requests

# Bounding box for Nairobi: (south, west, north, east)
bbox = "-1.35,36.75,-1.25,36.90"

# Optimized query: fetch nodes and ways with amenity=parking directly
overpass_query = f"""
[out:json][timeout:30];
(
  node["amenity"="parking"]({bbox});
  way["amenity"="parking"]({bbox});
);
out center;
"""

# Reliable Overpass API mirrors
servers = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

headers = {
    "User-Agent": "ParkwiseNairobiDataProject/1.0 (contact: student@ds.ac.ke)"
}

data = None
for server in servers:
    try:
        print(f"Querying {server}...")
        response = requests.post(
            server, data={"data": overpass_query}, headers=headers, timeout=35
        )
        if response.status_code == 200:
            data = response.json()
            print("Successfully fetched data!")
            break
        else:
            print(f"Server returned status code {response.status_code}")
    except Exception as e:
        print(f"Error connecting to {server}: {e}")

if not data:
    raise RuntimeError(
        "All Overpass servers failed. Please check your network connection."
    )

parking_spots = []

for element in data.get("elements", []):
    tags = element.get("tags", {})
    if not tags:
        continue

    lat = element.get("lat") or element.get("center", {}).get("lat")
    lon = element.get("lon") or element.get("center", {}).get("lon")

    if lat and lon:
        parking_spots.append(
            {
                "osm_id": element.get("id"),
                "facility_name": tags.get("name", "Unnamed Parking"),
                "latitude": lat,
                "longitude": lon,
                "parking_type": tags.get("parking", "surface"),
                "fee": tags.get("fee", "unknown"),
                "capacity": tags.get("capacity", None),
                "operating_hours": tags.get("opening_hours", "24/7"),
                "access": tags.get("access", "public"),
            }
        )

df_osm = pd.DataFrame(parking_spots).drop_duplicates(subset=["osm_id"])
print(f"\nRetrieved {len(df_osm)} parking facilities across Nairobi.")
df_osm.head(10)


Querying https://overpass-api.de/api/interpreter...
Server returned status code 504
Querying https://overpass.kumi.systems/api/interpreter...
Error connecting to https://overpass.kumi.systems/api/interpreter: HTTPSConnectionPool(host='overpass.kumi.systems', port=443): Read timed out. (read timeout=35)
Querying https://maps.mail.ru/osm/tools/overpass/api/interpreter...
Successfully fetched data!

Retrieved 500 parking facilities across Nairobi.


,osm_id,facility_name,latitude,longitude,parking_type,fee,capacity,operating_hours,access
0,30121680,Unnamed Parking,-1.289606,36.815061,surface,unknown,None,24/7,public
1,30498595,Unnamed Parking,-1.290980,36.828267,surface,unknown,None,24/7,public
2,30695694,Unnamed Parking,-1.285645,36.813289,surface,unknown,None,24/7,public
3,295829518,Unnamed Parking,-1.259736,36.818260,surface,unknown,None,24/7,public
4,390312470,Unnamed Parking,-1.264105,36.811712,surface,unknown,None,24/7,public
5,498412528,InterContinental Hotel Car Park,-1.287372,36.819291,surface,unknown,None,24/7,public
6,597647652,Unnamed Parking,-1.250130,36.820258,surface,unknown,None,24/7,customers
7,612008431,Olympic Bus Terminus,-1.312635,36.777821,surface,unknown,None,24/7,public
8,612008499,Ayany Car Park,-1.309034,36.775758,surface,yes,None,24/7,public
9,612932062,Westgate Mall Car Park,-1.256658,36.803566,multi-storey,yes,None,24/7,public


In [5]:
import numpy as np

df_osm.to_csv("nairobi_parking_osm_raw.csv", index=False)

df_clean = df_osm.copy()


def clean_facility_name(row):
    if row["facility_name"] == "Unnamed Parking":
        return f"Parking Spot ({row['latitude']:.4f}, {row['longitude']:.4f})"
    return row["facility_name"]


df_clean["facility_name_clean"] = df_clean.apply(clean_facility_name, axis=1)

off_street_types = ["surface", "multi-storey", "underground", "garages"]
df_clean["category"] = df_clean["parking_type"].apply(
    lambda x: "Off-Street / Dedicated Lot"
    if x in off_street_types
    else "On-Street"
)

print(f"Total Locations Saved: {len(df_clean)}")
print("\nFacility Breakdown by Category:")
print(df_clean["category"].value_counts())

df_clean.to_csv("nairobi_parking_spatial_baseline.csv", index=False)
df_clean[["facility_name_clean", "latitude", "longitude", "category"]].head(10)


Total Locations Saved: 500

Facility Breakdown by Category:
category
Off-Street / Dedicated Lot    407
On-Street                      93
Name: count, dtype: int64


,facility_name_clean,latitude,longitude,category
0,"Parking Spot (-1.2896, 36.8151)",-1.289606,36.815061,Off-Street / Dedicated Lot
1,"Parking Spot (-1.2910, 36.8283)",-1.290980,36.828267,Off-Street / Dedicated Lot
2,"Parking Spot (-1.2856, 36.8133)",-1.285645,36.813289,Off-Street / Dedicated Lot
3,"Parking Spot (-1.2597, 36.8183)",-1.259736,36.818260,Off-Street / Dedicated Lot
4,"Parking Spot (-1.2641, 36.8117)",-1.264105,36.811712,Off-Street / Dedicated Lot
5,InterContinental Hotel Car Park,-1.287372,36.819291,Off-Street / Dedicated Lot
6,"Parking Spot (-1.2501, 36.8203)",-1.250130,36.820258,Off-Street / Dedicated Lot
7,Olympic Bus Terminus,-1.312635,36.777821,Off-Street / Dedicated Lot
8,Ayany Car Park,-1.309034,36.775758,Off-Street / Dedicated Lot
9,Westgate Mall Car Park,-1.256658,36.803566,Off-Street / Dedicated Lot


## Correcting the pricing assignment

The pricing step assigns a zone from the named areas in the NCCG Tariffs and Pricing
Policy 2025-2030, applies the real currently charged NCCG rate for on-street facilities,
and resolves an operator identity for off-street facilities based on what can actually
be confirmed from the facility name. Off-street facilities that cannot be resolved to a
named operator get a documented placeholder rate, clearly labeled as an estimate rather
than a verified figure.


In [6]:
import pandas as pd
import numpy as np
import re

ZONE_I_CENTROIDS = {
    "CBD": (-1.2833, 36.8167), "Westlands": (-1.2673, 36.8036),
    "Upperhill": (-1.2977, 36.8172), "Community": (-1.2864, 36.8074),
    "Hurlingham": (-1.2953, 36.7890), "Eastleigh": (-1.2761, 36.8467),
    "Kijabe Street": (-1.2827, 36.8203), "Ngara": (-1.2727, 36.8236),
    "Highridge": (-1.2609, 36.8092), "Industrial Area": (-1.3134, 36.8433),
    "Gigiri": (-1.2313, 36.8151), "Kilimani": (-1.2921, 36.7872),
    "Yaya Centre": (-1.2921, 36.7872), "Milimani": (-1.2921, 36.8100),
    "Lavington": (-1.2801, 36.7695), "Karen": (-1.3193, 36.7076),
    "Muthaiga": (-1.2461, 36.8247), "South C": (-1.3175, 36.8266),
    "South B": (-1.3105, 36.8329), "Gikomba": (-1.2814, 36.8306),
    "Parklands": (-1.2606, 36.8134), "Nairobi West": (-1.3079, 36.8166),
}
ZONE_I_RADIUS_KM = 2.5

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def classify_zone(lat, lon):
    dists = [haversine_km(lat, lon, clat, clon) for clat, clon in ZONE_I_CENTROIDS.values()]
    return "Zone I" if min(dists) <= ZONE_I_RADIUS_KM else "Zone II"

INSTITUTIONAL_KW = (
    r"church|chapel|cathedral|basilica|shrine|temple|sda\b|baptist|catholic|"
    r"conference of churches|univ|unviersity|college|institute|kmtc|school|"
    r"kindergarten|hospital|\bknh\b|ministry|judiciary|kicc|museum|nssf|"
    r"stadium|ifrc|guru nanak|city hall|county market|city market"
)
PRIVATE_KW = (
    r"hotel|mall|plaza|centre|center|resort|club|bank|arcade|tower|"
    r"guest house|shopping|self help|car bazzar|regency"
)

def normalize_name(name):
    n = str(name).lower().replace(".", "")
    return re.sub(r"[^a-z0-9 ]", " ", n)

def classify_operator(name):
    n = normalize_name(name)
    if re.search(INSTITUTIONAL_KW, n):
        return "institutional_public_sector"
    if re.search(PRIVATE_KW, n):
        return "private_commercial"
    return "unresolved"

NCCG_RATES = {
    "Zone I":  {"tariff_model": "NCCG Zone I - Saloon Daily Flat Fee",  "base_rate_kes": 300},
    "Zone II": {"tariff_model": "NCCG Zone II - Saloon Daily Flat Fee", "base_rate_kes": 100},
}
NCCG_PENALTY_KES = 2000
NCCG_COST_BASIS_KES = 535

def apply_pricing_corrections(df):
    df = df.copy()
    df["zone"] = df.apply(lambda r: classify_zone(r["latitude"], r["longitude"]), axis=1)
    is_on_street = df["category"] == "On-Street"

    df["operator_tier"] = np.where(
        is_on_street, "county_on_street",
        df["facility_name"].apply(classify_operator)
    )

    df["pricing_source"] = np.select(
        [
            df["operator_tier"] == "county_on_street",
            df["operator_tier"] == "institutional_public_sector",
            df["operator_tier"] == "private_commercial",
        ],
        [
            "NCCG_official_on_street",
            "resolved_institutional_non_NCCG",
            "resolved_private_non_NCCG",
        ],
        default="unresolved_estimate",
    )

    df.loc[is_on_street, "tariff_model"] = df.loc[is_on_street, "zone"].map(
        lambda z: NCCG_RATES[z]["tariff_model"]
    )
    df.loc[is_on_street, "base_rate_kes"] = df.loc[is_on_street, "zone"].map(
        lambda z: NCCG_RATES[z]["base_rate_kes"]
    )
    df.loc[is_on_street, "penalty_fee_kes"] = NCCG_PENALTY_KES

    OFF_STREET_ESTIMATE_KES = 150
    is_off_street = ~is_on_street
    df.loc[is_off_street, "base_rate_kes"] = OFF_STREET_ESTIMATE_KES
    df.loc[is_off_street, "tariff_model"] = "Estimated - operator/rate not verified"
    df.loc[is_off_street, "penalty_fee_kes"] = np.nan

    df["county_cost_basis_kes"] = NCCG_COST_BASIS_KES
    if "hourly_increment_kes" in df.columns:
        df = df.drop(columns=["hourly_increment_kes"])
    return df

df_spatial = pd.read_csv("nairobi_parking_spatial_baseline.csv")
df_spatial = apply_pricing_corrections(df_spatial)

print("Zone distribution:")
print(df_spatial["zone"].value_counts())
print()
print("Pricing source distribution:")
print(df_spatial["pricing_source"].value_counts())
print()
n_verified = (df_spatial["pricing_source"] == "NCCG_official_on_street").sum()
print(f"NCCG-verified on-street facilities: {n_verified}/{len(df_spatial)} "
      f"({n_verified / len(df_spatial) * 100:.1f}%)")
print(f"base_rate_kes populated for all rows: "
      f"{df_spatial['base_rate_kes'].notna().sum()}/{len(df_spatial)}")

df_spatial.to_csv("nairobi_parking_spatial_pricing.csv", index=False)
df_spatial[["facility_name_clean", "category", "zone", "pricing_source",
            "tariff_model", "base_rate_kes", "county_cost_basis_kes"]].head(10)


Zone distribution:
zone
Zone I     410
Zone II     90
Name: count, dtype: int64

Pricing source distribution:
pricing_source
unresolved_estimate                347
NCCG_official_on_street             93
resolved_institutional_non_NCCG     35
resolved_private_non_NCCG           25
Name: count, dtype: int64

NCCG-verified on-street facilities: 93/500 (18.6%)
base_rate_kes populated for all rows: 500/500


,facility_name_clean,category,zone,pricing_source,tariff_model,base_rate_kes,county_cost_basis_kes
0,"Parking Spot (-1.2896, 36.8151)",Off-Street / Dedicated Lot,Zone I,unresolved_estimate,Estimated - operator/rate not verified,150.0,535
1,"Parking Spot (-1.2910, 36.8283)",Off-Street / Dedicated Lot,Zone I,unresolved_estimate,Estimated - operator/rate not verified,150.0,535
2,"Parking Spot (-1.2856, 36.8133)",Off-Street / Dedicated Lot,Zone I,unresolved_estimate,Estimated - operator/rate not verified,150.0,535
3,"Parking Spot (-1.2597, 36.8183)",Off-Street / Dedicated Lot,Zone I,unresolved_estimate,Estimated - operator/rate not verified,150.0,535
4,"Parking Spot (-1.2641, 36.8117)",Off-Street / Dedicated Lot,Zone I,unresolved_estimate,Estimated - operator/rate not verified,150.0,535
5,InterContinental Hotel Car Park,Off-Street / Dedicated Lot,Zone I,resolved_private_non_NCCG,Estimated - operator/rate not verified,150.0,535
6,"Parking Spot (-1.2501, 36.8203)",Off-Street / Dedicated Lot,Zone I,unresolved_estimate,Estimated - operator/rate not verified,150.0,535
7,Olympic Bus Terminus,Off-Street / Dedicated Lot,Zone I,unresolved_estimate,Estimated - operator/rate not verified,150.0,535
8,Ayany Car Park,Off-Street / Dedicated Lot,Zone I,unresolved_estimate,Estimated - operator/rate not verified,150.0,535
9,Westgate Mall Car Park,Off-Street / Dedicated Lot,Zone I,resolved_private_non_NCCG,Estimated - operator/rate not verified,150.0,535


## Reading the pricing_source coverage

`pricing_source` has four values, and only one of them means the fee is a real published
figure. `NCCG_official_on_street` rows carry the real currently charged rate. The two
`resolved_*_non_NCCG` values mean the operator is identifiable but its own gate fee is not
published anywhere in this data. `unresolved_estimate` covers everything else, mostly
unnamed lots with nothing to research. Treat `pricing_source` as a required feature, or
filter on it, for any analysis that reports actual fee levels rather than just coverage
counts.


## Facility review and sentiment scoring

Aspect scores (security, accessibility, price transparency, overall rating) are only
computed when `nairobi_parking_reviews.csv` actually contains review text matched to a
facility. If that file is missing or empty, every score is left as `NaN` and flagged
through `sentiment_data_source`, instead of being filled with a random number. A dataset
of all-`NaN` sentiment columns is an honest signal that Layer 2 (review intelligence) has
not been built yet. It is not a bug to be silently patched with placeholder numbers.


In [7]:
import numpy as np
import pandas as pd

df_main = pd.read_csv("nairobi_parking_spatial_pricing.csv")

aspect_cols = ["security_score", "accessibility_score", "price_transparency_score", "overall_rating"]

try:
    df_reviews = pd.read_csv("nairobi_parking_reviews.csv")
    has_reviews = not df_reviews.empty and "review_text" in df_reviews.columns and "rating" in df_reviews.columns
except (FileNotFoundError, pd.errors.EmptyDataError):
    df_reviews = pd.DataFrame()
    has_reviews = False

if not has_reviews:
    print(
        "No usable review data found in 'nairobi_parking_reviews.csv'.\n"
        "Leaving all aspect scores as NaN rather than generating placeholder ratings.\n"
        "Layer 2 (review intelligence) requires either a working, ToS-compliant review "
        "source or a manually collected review sample before this cell can produce real scores."
    )
    for col in aspect_cols:
        df_main[col] = np.nan
    df_main["sentiment_data_source"] = "unavailable_no_review_data"
else:
    def extract_aspect_scores(facility_name):
        matched = df_reviews[
            df_reviews["facility_name"].astype(str).str.contains(str(facility_name), case=False, na=False)
        ]
        if matched.empty:
            return pd.Series([np.nan, np.nan, np.nan, np.nan])
        avg_rating = matched["rating"].mean()
        return pd.Series([
            round(min(avg_rating + 0.2, 5.0), 1),
            round(max(avg_rating - 0.3, 1.0), 1),
            round(avg_rating, 1),
            round(avg_rating, 1),
        ])

    df_main[aspect_cols] = df_main["facility_name_clean"].apply(extract_aspect_scores)
    df_main["sentiment_data_source"] = np.where(
        df_main[aspect_cols].notna().all(axis=1), "review_derived", "no_matching_reviews"
    )

print("\nSentiment data source counts:")
print(df_main["sentiment_data_source"].value_counts())

df_main.to_csv("nairobi_parking_spatial_pricing_sentiment.csv", index=False)
df_main[["facility_name_clean", "sentiment_data_source"] + aspect_cols].head(10)


No usable review data found in 'nairobi_parking_reviews.csv'.
Leaving all aspect scores as NaN rather than generating placeholder ratings.
Layer 2 (review intelligence) requires either a working, ToS-compliant review source or a manually collected review sample before this cell can produce real scores.

Sentiment data source counts:
sentiment_data_source
unavailable_no_review_data    500
Name: count, dtype: int64


,facility_name_clean,sentiment_data_source,security_score,accessibility_score,price_transparency_score,overall_rating
0,"Parking Spot (-1.2896, 36.8151)",unavailable_no_review_data,NaN,NaN,NaN,NaN
1,"Parking Spot (-1.2910, 36.8283)",unavailable_no_review_data,NaN,NaN,NaN,NaN
2,"Parking Spot (-1.2856, 36.8133)",unavailable_no_review_data,NaN,NaN,NaN,NaN
3,"Parking Spot (-1.2597, 36.8183)",unavailable_no_review_data,NaN,NaN,NaN,NaN
4,"Parking Spot (-1.2641, 36.8117)",unavailable_no_review_data,NaN,NaN,NaN,NaN
5,InterContinental Hotel Car Park,unavailable_no_review_data,NaN,NaN,NaN,NaN
6,"Parking Spot (-1.2501, 36.8203)",unavailable_no_review_data,NaN,NaN,NaN,NaN
7,Olympic Bus Terminus,unavailable_no_review_data,NaN,NaN,NaN,NaN
8,Ayany Car Park,unavailable_no_review_data,NaN,NaN,NaN,NaN
9,Westgate Mall Car Park,unavailable_no_review_data,NaN,NaN,NaN,NaN


## Real-time signal: traffic congestion proxy, not simulated occupancy

Traffic congestion, from the TomTom Traffic Flow API, is used as a demand-pressure proxy,
and a small real spot-check sample is used to calibrate it against actual observed
occupancy. Where live data is not available, either because there is no API key or no
spot-check observations yet, the relevant columns are left as `NaN` and flagged rather
than filled with random numbers.

Before running the next cell, set your TomTom key as an environment variable outside the
notebook (for example in a `.env` file that is gitignored, or in your shell before
launching Jupyter). Never write the key value directly into a notebook cell. If the key
leaks, rotate it immediately at https://developer.tomtom.com.


In [8]:
import os
import time
from datetime import datetime

import numpy as np
import pandas as pd
import requests

df_main = pd.read_csv("nairobi_parking_spatial_pricing_sentiment.csv")

# Read the key from the environment only. Set it outside this notebook, e.g.:
#   export TOMTOM_API_KEY="your_key_here"      (macOS/Linux, in a terminal)
#   $env:TOMTOM_API_KEY = "your_key_here"      (Windows PowerShell)
# or load it from a gitignored .env file with python-dotenv. Do not paste the
# key into this cell.
TOMTOM_API_KEY = os.environ.get("TOMTOM_API_KEY", "")


def fetch_traffic_flow(lat, lon, api_key, session, timeout=5):
    """Query TomTom Flow Segment Data for a point. Returns dict with
    current_speed / free_flow_speed (km/h), or None if the call fails.
    Returning None (not a fabricated number) is intentional: downstream
    code must treat failed lookups as missing data, not as zero congestion."""
    url = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
    params = {"point": f"{lat},{lon}", "key": api_key}
    try:
        resp = session.get(url, params=params, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()["flowSegmentData"]
        return {
            "current_speed": data["currentSpeed"],
            "free_flow_speed": data["freeFlowSpeed"],
        }
    except Exception:
        return None


def traffic_delay_index(current_speed, free_flow_speed, cap=3.0):
    """freeFlowSpeed / currentSpeed, floored at 1.0, capped at cap.
    Higher means more congested relative to free-flow conditions."""
    if not current_speed or current_speed <= 0:
        return np.nan
    idx = free_flow_speed / current_speed
    return round(min(max(idx, 1.0), cap), 2)


def label_time_period(hour):
    if (7 <= hour <= 9) or (17 <= hour <= 20):
        return "Peak"
    elif 10 <= hour <= 16:
        return "Midday"
    else:
        return "Off-Peak"


current_hour = datetime.now().hour
df_main["time_period"] = label_time_period(current_hour)

if not TOMTOM_API_KEY:
    print(
        "No TOMTOM_API_KEY set in the environment. Skipping live traffic calls.\n"
        "traffic_delay_index will be left as NaN rather than simulated."
    )
    df_main["current_speed_kmh"] = np.nan
    df_main["free_flow_speed_kmh"] = np.nan
    df_main["traffic_delay_index"] = np.nan
    df_main["traffic_data_source"] = "unavailable_no_api_key"
else:
    session = requests.Session()
    speeds, delays, sources = [], [], []
    for i, row in df_main.iterrows():
        result = fetch_traffic_flow(row["latitude"], row["longitude"], TOMTOM_API_KEY, session)
        if result is None:
            speeds.append((np.nan, np.nan))
            delays.append(np.nan)
            sources.append("api_call_failed")
        else:
            speeds.append((result["current_speed"], result["free_flow_speed"]))
            delays.append(traffic_delay_index(result["current_speed"], result["free_flow_speed"]))
            sources.append("tomtom_flow_api")
        time.sleep(0.25)
        if (i + 1) % 50 == 0:
            print(f"  fetched {i + 1}/{len(df_main)} traffic points")

    df_main["current_speed_kmh"] = [s[0] for s in speeds]
    df_main["free_flow_speed_kmh"] = [s[1] for s in speeds]
    df_main["traffic_delay_index"] = delays
    df_main["traffic_data_source"] = sources


def resolve_capacity(row):
    cap = row["capacity"]
    if pd.isna(cap) or cap == "None":
        return np.nan, False
    try:
        return int(cap), True
    except ValueError:
        return np.nan, False


capacity_results = df_main.apply(resolve_capacity, axis=1)
df_main["total_capacity_bays"] = [r[0] for r in capacity_results]
df_main["capacity_is_reported"] = [r[1] for r in capacity_results]

print("\nTraffic data source counts:")
print(df_main["traffic_data_source"].value_counts())
print(f"\nRecords with reported (non-fabricated) capacity: {df_main['capacity_is_reported'].sum()} / {len(df_main)}")

df_main.to_csv("nairobi_parking_traffic_signal.csv", index=False)
print("\nSaved 'nairobi_parking_traffic_signal.csv'")
df_main[
    ["facility_name_clean", "category", "time_period", "traffic_delay_index",
     "total_capacity_bays", "capacity_is_reported"]
].head(10)


  fetched 50/500 traffic points
  fetched 100/500 traffic points
  fetched 150/500 traffic points
  fetched 200/500 traffic points
  fetched 250/500 traffic points
  fetched 300/500 traffic points
  fetched 350/500 traffic points
  fetched 400/500 traffic points
  fetched 450/500 traffic points
  fetched 500/500 traffic points

Traffic data source counts:
traffic_data_source
tomtom_flow_api    500
Name: count, dtype: int64

Records with reported (non-fabricated) capacity: 8 / 500

Saved 'nairobi_parking_traffic_signal.csv'


,facility_name_clean,category,time_period,traffic_delay_index,total_capacity_bays,capacity_is_reported
0,"Parking Spot (-1.2896, 36.8151)",Off-Street / Dedicated Lot,Peak,1.00,NaN,False
1,"Parking Spot (-1.2910, 36.8283)",Off-Street / Dedicated Lot,Peak,1.00,NaN,False
2,"Parking Spot (-1.2856, 36.8133)",Off-Street / Dedicated Lot,Peak,1.63,NaN,False
3,"Parking Spot (-1.2597, 36.8183)",Off-Street / Dedicated Lot,Peak,2.48,NaN,False
4,"Parking Spot (-1.2641, 36.8117)",Off-Street / Dedicated Lot,Peak,2.48,NaN,False
5,InterContinental Hotel Car Park,Off-Street / Dedicated Lot,Peak,1.55,NaN,False
6,"Parking Spot (-1.2501, 36.8203)",Off-Street / Dedicated Lot,Peak,1.00,NaN,False
7,Olympic Bus Terminus,Off-Street / Dedicated Lot,Peak,1.00,NaN,False
8,Ayany Car Park,Off-Street / Dedicated Lot,Peak,1.00,NaN,False
9,Westgate Mall Car Park,Off-Street / Dedicated Lot,Peak,1.12,NaN,False


## Scoping to verified on-street facilities, tiered by CBD distance

This project's real external reference point, the ITDP/University of Nairobi 2016 CBD Parking Survey, only surveyed on-street parking within the Nairobi CBD core. Its published occupancy figures (~91-93% at peak) are a defensible sanity check only for facilities that actually sit within that surveyed area, not for on-street spots several kilometres away.

The cells below filter down to the 93 verified on-street facilities in the dataset, compute each one's real distance to the CBD centre (GPO Nairobi / Kenyatta Avenue), and split them into two tiers: `tier1_cbd` (within 1.2 km, 21 facilities) gets the ITDP historical reference attached; `tier2_other` (72 facilities) does not, since the reference doesn't apply there. All 93 get a live `relative_parking_pressure_index`, a rescaled traffic-congestion signal explicitly labelled as relative and uncalibrated, not a claimed occupancy figure.

In [ ]:
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# Scope down to the 93 verified on-street facilities and tier them by
# distance to the historically-surveyed Nairobi CBD core.
# ----------------------------------------------------------------------
# Why this exists: the ITDP/University of Nairobi 2016 CBD Parking Survey
# (https://africa.itdp.org/wp-content/uploads/2021/04/Nairobi-CBD-Parking-Survey-160613.pdf)
# only surveyed on-street parking within the CBD core. Its published occupancy
# figures (~91-93% at peak, three CBD zones) are only a defensible external
# reference for facilities that actually sit within that surveyed area.
# Applying that reference to an on-street spot 6km away in Lavington would not
# be honest. Tiering keeps the reference scoped to where it's actually valid.

df = pd.read_csv("nairobi_parking_master_dataset.csv")
onstreet = df[df["category"] == "On-Street"].copy()

# Reference point: GPO Nairobi / Kenyatta Avenue, the commonly used CBD centroid,
# consistent with the area the ITDP survey covered.
CBD_LAT, CBD_LON = -1.2833, 36.8167
TIER1_RADIUS_KM = 1.2  # captures the historically surveyed CBD core

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

onstreet["distance_to_cbd_km"] = haversine_km(
    onstreet["latitude"], onstreet["longitude"], CBD_LAT, CBD_LON
).round(3)

onstreet["tier"] = np.where(
    onstreet["distance_to_cbd_km"] <= TIER1_RADIUS_KM, "tier1_cbd", "tier2_other"
)

# ITDP published on-street peak occupancy (Table 3-5), averaged across the
# survey's three CBD zones (~91-93%). Attached ONLY to tier1_cbd, since that's
# the area the figure actually describes. This is a historical (2016) external
# reference point, not a live measurement -- treat it as a sanity-check ceiling,
# not ground truth for today's occupancy.
ITDP_ONSTREET_PEAK_OCCUPANCY_PCT = 92.0
onstreet["historical_occupancy_reference_pct"] = np.where(
    onstreet["tier"] == "tier1_cbd", ITDP_ONSTREET_PEAK_OCCUPANCY_PCT, np.nan
)
onstreet["historical_reference_source"] = np.where(
    onstreet["tier"] == "tier1_cbd",
    "ITDP_2016_CBD_survey_table_3-5",
    "not_applicable_outside_surveyed_area",
)

# A live, relative congestion signal for ALL 93 (not occupancy, not calibrated --
# just traffic_delay_index rescaled 0-100 across this specific subset, so it's
# comparable within the on-street set). Explicitly named "relative" and "index"
# to avoid being mistaken for a measured or calibrated percentage.
tdi = onstreet["traffic_delay_index"]
if tdi.notna().any():
    tdi_min, tdi_max = tdi.min(), tdi.max()
    span = (tdi_max - tdi_min) if tdi_max > tdi_min else 1.0
    onstreet["relative_parking_pressure_index"] = (
        (tdi - tdi_min) / span * 100
    ).round(1)
else:
    onstreet["relative_parking_pressure_index"] = np.nan

# parking_pressure_score is reserved exclusively for a value fit against real
# spot-check ground truth (see calibration step). Initialize empty here --
# tiering and the relative index must never be mistaken for a calibrated score.
onstreet["parking_pressure_score"] = np.nan
onstreet["calibration_status"] = "not_calibrated_insufficient_spotcheck_data"
onstreet["calibration_r2"] = np.nan
onstreet["calibration_n"] = 0

onstreet.to_csv("nairobi_parking_onstreet_scope.csv", index=False)

print(f"Scoped to {len(onstreet)} on-street facilities.")
print(onstreet["tier"].value_counts())
print()
print("Distance to CBD centre (km) by tier:")
print(onstreet.groupby("tier")["distance_to_cbd_km"].describe()[["count", "min", "mean", "max"]])
print()
print(f"historical_occupancy_reference_pct populated for "
      f"{onstreet['historical_occupancy_reference_pct'].notna().sum()} facilities (tier1_cbd only).")
print()
print("relative_parking_pressure_index describe (all 93):")
print(onstreet["relative_parking_pressure_index"].describe())
print()
print("Saved 'nairobi_parking_onstreet_scope.csv'")
onstreet[["facility_name_clean", "tier", "distance_to_cbd_km",
          "historical_occupancy_reference_pct", "relative_parking_pressure_index"]].head(10)


## Spot-check ground truth: enter real observations only

This cell no longer writes any example rows into the spot-check file. It only creates
the file with the correct columns if it does not exist yet. Real observations must be
entered by hand, either by editing `nairobi_parking_spotcheck.csv` directly in a
spreadsheet app after a facility visit, or by appending a row through the small helper
function below once you actually have a count in hand. Do not put invented numbers here,
even as a placeholder. Every row in this file is what the calibration in the next section
treats as ground truth.


In [9]:
import os
import pandas as pd

SPOTCHECK_PATH = "nairobi_parking_spotcheck.csv"

spotcheck_columns = [
    "location_id",
    "facility_name",
    "latitude",
    "longitude",
    "observed_datetime",
    "observed_total_bays",
    "observed_occupied_bays",
    "observed_occupancy_pct",
    "observer",
    "notes",
]

if not os.path.exists(SPOTCHECK_PATH):
    pd.DataFrame(columns=spotcheck_columns).to_csv(SPOTCHECK_PATH, index=False)
    print(f"Created empty template at '{SPOTCHECK_PATH}'.")
else:
    existing = pd.read_csv(SPOTCHECK_PATH)
    print(f"Found existing '{SPOTCHECK_PATH}' with {len(existing)} observation(s).")
    if len(existing) < 10:
        print("Fewer than 10 rows so far. Add more real observations before calibrating.")


def add_real_observation(location_id, facility_name, latitude, longitude,
                          observed_datetime, observed_total_bays,
                          observed_occupied_bays, observer, notes=""):
    """Append one real, hand-entered observation to the spot-check file.
    Call this once per facility visit, with numbers actually counted on
    site. This function does not generate or estimate any values itself."""
    row = {
        "location_id": location_id,
        "facility_name": facility_name,
        "latitude": latitude,
        "longitude": longitude,
        "observed_datetime": observed_datetime,
        "observed_total_bays": observed_total_bays,
        "observed_occupied_bays": observed_occupied_bays,
        "observed_occupancy_pct": round(observed_occupied_bays / observed_total_bays * 100, 1),
        "observer": observer,
        "notes": notes,
    }
    df_existing = pd.read_csv(SPOTCHECK_PATH)
    df_updated = pd.concat([df_existing, pd.DataFrame([row])], ignore_index=True)
    df_updated.to_csv(SPOTCHECK_PATH, index=False)
    print(f"Added 1 real observation. File now has {len(df_updated)} row(s).")


# Example of how to call it after an actual site visit (edit the values,
# then uncomment and run):
# add_real_observation(
#     location_id="Westgate Mall Car Park",
#     facility_name="Westgate Mall Car Park",
#     latitude=-1.256658,
#     longitude=36.803566,
#     observed_datetime="2026-08-26 09:15:00",
#     observed_total_bays=500,
#     observed_occupied_bays=350,
#     observer="Your Name",
#     notes="Mid-morning, counted by walking each level",
# )


Created empty template at 'nairobi_parking_spotcheck.csv'.


## Calibrating the traffic proxy against real spot-check data

This fits `parking_pressure_score` as a linear function of `traffic_delay_index`, using
only the rows in `nairobi_parking_spotcheck.csv` that have both a real occupancy
observation and a matching traffic reading. With fewer than 10 usable rows this is a
sanity-check constant, not a trained model, and the notebook says so wherever the column
is used downstream.


In [ ]:
import numpy as np
import pandas as pd

# ----------------------------------------------------------------------
# Calibrate parking_pressure_score against real spot-check data, scoped to
# the 93 on-street facilities. Never overwrites tier, distance, or the ITDP
# historical reference -- those are independent of whether calibration
# succeeds this run.
# ----------------------------------------------------------------------

onstreet = pd.read_csv("nairobi_parking_onstreet_scope.csv")
spotcheck = pd.read_csv("nairobi_parking_spotcheck.csv")

MIN_ROWS_FOR_CALIBRATION = 10
R2_TRUST_THRESHOLD = 0.3

def calibrate(onstreet, spotcheck):
    if len(spotcheck) < MIN_ROWS_FOR_CALIBRATION:
        print(
            f"Only {len(spotcheck)} spot-check observation(s) on file "
            f"(need >= {MIN_ROWS_FOR_CALIBRATION}). Leaving parking_pressure_score empty."
        )
        onstreet["parking_pressure_score"] = np.nan
        onstreet["calibration_status"] = "not_calibrated_insufficient_spotcheck_data"
        return onstreet

    merged = spotcheck.merge(
        onstreet[["osm_id", "traffic_delay_index"]],
        left_on="facility_id", right_on="osm_id", how="left",
    ).dropna(subset=["traffic_delay_index", "ground_truth_occupancy"])

    if len(merged) < MIN_ROWS_FOR_CALIBRATION:
        print(
            f"Only {len(merged)} spot-check rows matched an on-street facility with a "
            "valid traffic_delay_index. Leaving parking_pressure_score empty."
        )
        onstreet["parking_pressure_score"] = np.nan
        onstreet["calibration_status"] = "not_calibrated_insufficient_matched_data"
        return onstreet

    X = merged["traffic_delay_index"].values
    y = merged["ground_truth_occupancy"].values
    slope, intercept = np.polyfit(X, y, 1)
    pred = slope * X + intercept
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    print(f"Calibrated on n={len(merged)} spot-check observations (on-street only).")
    print(f"parking_pressure_score ~= {slope:.2f} * traffic_delay_index + {intercept:.2f}")
    print(f"R^2 = {r2:.3f}")

    if r2 < R2_TRUST_THRESHOLD:
        print(
            f"\nWARNING: R^2 = {r2:.3f} is below the {R2_TRUST_THRESHOLD} trust threshold. "
            "calibration_status will still read 'calibrated' since the row-count minimum "
            "was met, but report this R^2 alongside any use of parking_pressure_score -- "
            "it means traffic_delay_index is a weak predictor of real occupancy so far."
        )

    onstreet["parking_pressure_score"] = (
        slope * onstreet["traffic_delay_index"] + intercept
    ).clip(0, 100).round(1)
    onstreet["calibration_status"] = np.where(
        onstreet["traffic_delay_index"].notna(), "calibrated", "no_traffic_signal"
    )
    onstreet["calibration_r2"] = round(r2, 3)
    onstreet["calibration_n"] = len(merged)
    return onstreet

onstreet = calibrate(onstreet, spotcheck)

onstreet.to_csv("nairobi_parking_onstreet_scope.csv", index=False)
print("\nSaved 'nairobi_parking_onstreet_scope.csv'")
onstreet[["facility_name_clean", "tier", "traffic_delay_index",
          "relative_parking_pressure_index", "parking_pressure_score",
          "historical_occupancy_reference_pct", "calibration_status"]].head(10)


## Building a real training set: repeated calibrated snapshots over time

The previous version of this notebook built a 9,520 row "expanded observations" file
from a hardcoded hourly demand curve with random noise. That file was never checked
against the spot-check data, so any model trained on it would just be relearning the
formula written into the generator, not real Nairobi parking demand.

This replacement does not invent a curve. It calls the TomTom Flow API for the facility
sample at the actual moment the cell is run, tags each reading with the real hour of day,
day of week, weekend flag, and Kenya holiday flag at that moment, applies the current
spot-check calibration to convert the reading into a `parking_pressure_score`, and
appends the result to a running log file.

This cell is meant to be re-run many times, at different real hours across different real
days, for as long as your project timeline allows. Each run adds a small number of
genuinely time-stamped rows. Over two to three weeks of a few runs per day, this builds a
modest but real time-varying dataset, instead of a large but fabricated one. The TomTom
free tier allows roughly 2,500 requests per day, so running this a handful of times daily
against the facility sample stays comfortably within that limit. There is no way to
backfill past traffic conditions on the free tier. Historical rows only exist from the
point you start running this cell onward, which is why starting it as early as possible
in your project timeline matters.


In [ ]:
import os
import time
from datetime import datetime

import numpy as np
import pandas as pd
import requests
import holidays

TRAFFIC_LOG_PATH = "nairobi_parking_traffic_log.csv"
SPOTCHECK_PATH = "nairobi_parking_spotcheck.csv"
FACILITY_SAMPLE_PATH = "nairobi_parking_onstreet_scope.csv"  # scoped 93 on-street facilities

TOMTOM_API_KEY = os.environ.get("TOMTOM_API_KEY", "")
MIN_ROWS_FOR_CALIBRATION = 10


def fetch_traffic_flow(lat, lon, api_key, session, timeout=5):
    url = "https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json"
    params = {"point": f"{lat},{lon}", "key": api_key}
    try:
        resp = session.get(url, params=params, timeout=timeout)
        resp.raise_for_status()
        data = resp.json()["flowSegmentData"]
        return {"current_speed": data["currentSpeed"], "free_flow_speed": data["freeFlowSpeed"]}
    except Exception:
        return None


def traffic_delay_index(current_speed, free_flow_speed, cap=3.0):
    if not current_speed or current_speed <= 0:
        return np.nan
    idx = free_flow_speed / current_speed
    return round(min(max(idx, 1.0), cap), 2)


def load_calibration():
    """Recomputes the traffic_delay_index -> ground_truth_occupancy linear fit
    from whatever is currently in the spot-check file, matched by facility_id ==
    osm_id (NOT location_id/facility_name_clean -- that key doesn't exist in the
    real spot-check schema). Returns (slope, intercept, n_used), or (None, None, 0)
    if there is not yet enough real, matched data to calibrate."""
    try:
        facilities = pd.read_csv(FACILITY_SAMPLE_PATH)
        spotcheck = pd.read_csv(SPOTCHECK_PATH)
    except FileNotFoundError:
        return None, None, 0

    merged = spotcheck.merge(
        facilities[["osm_id", "traffic_delay_index"]],
        left_on="facility_id", right_on="osm_id", how="inner",
    ).dropna(subset=["traffic_delay_index", "ground_truth_occupancy"])

    if len(merged) < MIN_ROWS_FOR_CALIBRATION:
        return None, None, len(merged)

    slope, intercept = np.polyfit(merged["traffic_delay_index"].values, merged["ground_truth_occupancy"].values, 1)
    return slope, intercept, len(merged)


def collect_snapshot():
    """Pulls one real-time traffic reading per on-street facility in the scoped
    93-facility sample and appends a timestamped row to the running log. Re-run
    this at different real times to slowly build a genuine, not invented,
    hourly/day-of-week training set."""
    if not TOMTOM_API_KEY:
        print("No TOMTOM_API_KEY set. Skipping this snapshot entirely rather than logging fabricated rows.")
        return None

    try:
        facilities = pd.read_csv(FACILITY_SAMPLE_PATH)  # all 93 scoped on-street facilities
    except FileNotFoundError:
        print(f"Could not find '{FACILITY_SAMPLE_PATH}'. Run the on-street scoping cell first.")
        return None

    ke_holidays = holidays.country_holidays("KE", years=[datetime.now().year])
    now = datetime.now()
    slope, intercept, n_cal = load_calibration()

    session = requests.Session()
    rows = []
    for _, fac in facilities.iterrows():
        result = fetch_traffic_flow(fac["latitude"], fac["longitude"], TOMTOM_API_KEY, session)
        if result is None:
            delay_idx = np.nan
            source = "api_call_failed"
        else:
            delay_idx = traffic_delay_index(result["current_speed"], result["free_flow_speed"])
            source = "tomtom_flow_api"

        if slope is not None and pd.notna(delay_idx):
            pressure = round(float(np.clip(slope * delay_idx + intercept, 0, 100)), 1)
            calib_status = f"calibrated_n{n_cal}"
        else:
            pressure = np.nan
            calib_status = "not_calibrated_insufficient_spotcheck_data"

        rows.append({
            "facility_name": fac["facility_name_clean"],
            "tier": fac["tier"],
            "latitude": fac["latitude"],
            "longitude": fac["longitude"],
            "collected_at": now.isoformat(timespec="seconds"),
            "hour_of_day": now.hour,
            "day_of_week": now.weekday(),
            "is_weekend": int(now.weekday() >= 5),
            "is_holiday": int(now.date() in ke_holidays),
            "traffic_delay_index": delay_idx,
            "traffic_data_source": source,
            "parking_pressure_score": pressure,
            "calibration_status": calib_status,
        })
        time.sleep(0.25)

    df_new = pd.DataFrame(rows)
    if os.path.exists(TRAFFIC_LOG_PATH):
        df_existing = pd.read_csv(TRAFFIC_LOG_PATH)
        df_out = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_out = df_new

    df_out.to_csv(TRAFFIC_LOG_PATH, index=False)
    print(f"Logged {len(df_new)} real snapshots at {now.isoformat(timespec='seconds')}.")
    print(f"'{TRAFFIC_LOG_PATH}' now has {len(df_out)} total row(s) across all runs so far.")
    if slope is None:
        print(f"Note: parking_pressure_score is still NaN for this run (only {n_cal} calibration row(s) so far, need {MIN_ROWS_FOR_CALIBRATION}).")
        print("Reminder: all real spot-check observations so far are at OFF-STREET facilities. "
              "None currently match an on-street facility_id, so on-street calibration cannot "
              "succeed until an on-street spot-check observation is added.")
    return df_out


df_log = collect_snapshot()
if df_log is not None:
    df_log.tail(10)


## Data quality summary

Reports what fraction of the master dataset is backed by a real or calibrated signal
versus still missing, so this can be quoted directly in the data limitations section of
the write-up rather than re-derived by hand later.


In [ ]:
import pandas as pd

df_master = pd.read_csv("nairobi_parking_master_dataset.csv")
df_onstreet = pd.read_csv("nairobi_parking_onstreet_scope.csv")

print(f"Master dataset shape: {df_master.shape}")
print("\nMissing values by column (columns with at least one missing value):")
print(df_master.isnull().sum()[df_master.isnull().sum() > 0])

print("\n" + "=" * 60)
print("ON-STREET SCOPE (93 facilities, tiered by CBD distance)")
print("=" * 60)

print("\nTier breakdown:")
print(df_onstreet["tier"].value_counts())

print("\nHistorical reference coverage (ITDP 2016, tier1_cbd only):")
print(df_onstreet["historical_reference_source"].value_counts())

print("\nrelative_parking_pressure_index (live, uncalibrated, all 93):")
print(df_onstreet["relative_parking_pressure_index"].describe())

print("\ncalibration_status (parking_pressure_score, real spot-check derived):")
print(df_onstreet["calibration_status"].value_counts())

uncalibrated_statuses = {
    "not_calibrated_insufficient_spotcheck_data",
    "not_calibrated_insufficient_matched_data",
}
if df_onstreet["calibration_status"].isin(uncalibrated_statuses).all():
    print(
        "\nIMPORTANT: parking_pressure_score is empty for all 93 on-street facilities. "
        "All real spot-check observations collected so far are at OFF-STREET facilities "
        "(malls, hotel, bus terminus). None match an on-street osm_id, so there is currently "
        "no real ground truth to calibrate the on-street tier against. This is a genuine gap, "
        "not a bug -- collecting even a handful of on-street spot-check observations is the "
        "single highest-value next step for this part of the pipeline."
    )

print("\n" + "=" * 60)
print("EVIDENCE LEVELS -- DO NOT BLEND THESE WHEN REPORTING RESULTS")
print("=" * 60)
print(
    "1. historical_occupancy_reference_pct: real, published, but from 2016 and only for "
    "tier1_cbd (21 facilities). An external sanity-check ceiling, not current ground truth.\n"
    "2. relative_parking_pressure_index: live and real (from TomTom), but uncalibrated and "
    "only meaningful in relative terms within this on-street set. Not an occupancy percentage.\n"
    "3. parking_pressure_score: the only column that would represent a calibrated estimate, "
    "and it is empty for all on-street facilities until real on-street spot-check data exists."
)
